# 😀 Real-Time Facial Emotion Detector — CNN + Transfer Learning
**Author:** Yuvraj Agrawal  
**Dataset:** FER-2013 (35,887 grayscale images, 7 emotion classes)  
**Stack:** Python, TensorFlow/Keras, OpenCV, NumPy, Matplotlib  
**Goal:** Train a CNN to classify 7 facial emotions; build a real-time webcam inference pipeline

### 7 Emotion Classes
| Label | Emotion  |
|-------|----------|
| 0     | Angry    |
| 1     | Disgust  |
| 2     | Fear     |
| 3     | Happy    |
| 4     | Sad      |
| 5     | Surprise |
| 6     | Neutral  |

---
## Project Structure
1. Setup & Kaggle API Configuration
2. Download & Extract FER-2013 Dataset
3. Explore & Visualise the Data
4. Preprocess & Build Data Pipelines
5. Build CNN Architecture (Custom + Transfer Learning)
6. Train with Class Weights & Callbacks
7. Evaluate — Accuracy, Loss, Confusion Matrix
8. Real-Time Webcam Emotion Detection
9. Save Model & Project Summary

In [ ]:
#Downloading Kaggle dataset
import os
os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_token = '{"username":"YOUR_KAGGLE_USERNAME","key":"YOUR_API_KEY_HERE"}'

with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(kaggle_token)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle configured ✓')

In [ ]:
!kaggle datasets download -d msambare/fer2013

## Step 1 — Setup & Imports

In [ ]:
!pip install opencv-python-headless -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os, cv2, zipfile

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

EMOTION_LABELS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
NUM_CLASSES    = 7
IMG_SIZE       = 48   # FER-2013 images are 48x48 pixels

print('TensorFlow:', tf.__version__)
print('OpenCV    :', cv2.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))
print('Setup complete ✓')

## Step 2 — Download FER-2013 Dataset
FER-2013 is on Kaggle. We use the Kaggle API to download it directly into Colab.
**You need a free Kaggle account.** Follow the instructions in the cell below.

In [ ]:
# Download FER-2013 dataset
!kaggle datasets download -d msambare/fer2013

# Extracting
with zipfile.ZipFile('fer2013.zip', 'r') as z:
    z.extractall('/content/fer2013/')

print('Dataset extracted ✓')
print('Folder structure:')
for root, dirs, files_list in os.walk('/content/fer2013'):
    level = root.replace('/content/fer2013', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files_list[:3]:
            print(f'{indent}  {f}')

In [ ]:
# Set dataset paths

TRAIN_DIR = '/content/fer2013/train'
TEST_DIR  = '/content/fer2013/test'

# Count images per class
print('=== Dataset Statistics ===')
print(f'{"Emotion":<12} {"Train":>8} {"Test":>8}')
print('-' * 30)
total_train, total_test = 0, 0
for emotion in sorted(os.listdir(TRAIN_DIR)):
    train_count = len(os.listdir(os.path.join(TRAIN_DIR, emotion)))
    test_count  = len(os.listdir(os.path.join(TEST_DIR,  emotion)))
    total_train += train_count
    total_test  += test_count
    print(f'{emotion:<12} {train_count:>8} {test_count:>8}')
print('-' * 30)
print(f'{"TOTAL":<12} {total_train:>8} {total_test:>8}')
print('\nNOTE: Dataset is imbalanced — Disgust has far fewer samples.')
print('We will handle this with class weights during training.')

## Step 3 — Explore & Visualise the Data

In [ ]:
# ── VISUALISE SAMPLE IMAGES PER EMOTION ──────────────────────
fig, axes = plt.subplots(3, 7, figsize=(16, 7))
fig.suptitle('FER-2013 Sample Images — 3 Samples per Emotion Class', fontsize=13, fontweight='bold')

for col, emotion in enumerate(sorted(os.listdir(TRAIN_DIR))):
    emotion_path = os.path.join(TRAIN_DIR, emotion)
    images = os.listdir(emotion_path)[:3]
    for row, img_name in enumerate(images):
        img_path = os.path.join(emotion_path, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        axes[row, col].imshow(img, cmap='gray')
        if row == 0:
            axes[row, col].set_title(emotion, fontsize=9, fontweight='bold')
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('sample_emotions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Observation: Fear and Sad look visually similar — model will struggle with these.')

In [ ]:
# ── CLASS DISTRIBUTION BAR CHART ─────────────────────────────
train_counts = []
emotions_sorted = sorted(os.listdir(TRAIN_DIR))
for emotion in emotions_sorted:
    train_counts.append(len(os.listdir(os.path.join(TRAIN_DIR, emotion))))

colors = ['#E24B4A','#D85A30','#BA7517','#639922','#1D9E75','#378ADD','#7F77DD']
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(emotions_sorted, train_counts, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Class Distribution — Training Set', fontsize=13, fontweight='bold')
ax.set_xlabel('Emotion')
ax.set_ylabel('Number of Images')
for bar, count in zip(bars, train_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', va='bottom', fontsize=10)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Disgust has ~600 images vs Happy ~8,000 — severe imbalance!')
print('Solution: compute class weights so rare classes are penalised more during training.')

## Step 4 — Preprocess & Build Data Pipelines

In [ ]:
# ── DATA GENERATORS ──────────────────────────────────────────
# ImageDataGenerator handles:
#   - Loading images from folder structure automatically
#   - Resizing to IMG_SIZE x IMG_SIZE
#   - Normalising pixels to [0, 1]
#   - Augmenting training images to reduce overfitting

# Training generator WITH augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0/255,           # Normalise: [0,255] → [0,1]
    rotation_range=15,         # Random rotation ±15°
    zoom_range=0.15,           # Random zoom ±15%
    width_shift_range=0.15,    # Random horizontal shift
    height_shift_range=0.15,   # Random vertical shift
    horizontal_flip=True,      # Mirror image left-right
    fill_mode='nearest',       # Fill empty pixels after shift
    validation_split=0.15      # Reserve 15% for validation
)

# Test generator WITHOUT augmentation
test_datagen = ImageDataGenerator(rescale=1.0/255)

BATCH_SIZE = 64

# Training set
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

# Validation set
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

# Test set
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('\nClass index mapping:')
print(train_generator.class_indices)

In [ ]:
# ── COMPUTE CLASS WEIGHTS ─────────────────────────────────────

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights_array))

print('Class weights (higher = rarer class, penalised more):')
for idx, emotion in enumerate(emotions_sorted):
    print(f'  {emotion:<10}: {class_weight_dict[idx]:.3f}')

## Step 5 — Build the CNN Architecture
We build TWO models and compare:
- **Model A**: Custom CNN trained from scratch
- **Model B**: Transfer Learning using VGG16 pre-trained on ImageNet

We train both and use whichever performs better.

In [ ]:
# ── MODEL A: CUSTOM CNN ───────────────────────────────────────
# Architecture: 3 conv blocks + dense head
# Input: (48, 48, 1) grayscale

def build_custom_cnn():
    model = models.Sequential([

        # Block 1 — detect basic edges and textures
        layers.Conv2D(64, (3,3), padding='same', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2,2),  # 48 → 24
        layers.Dropout(0.25),

        # Block 2 — detect facial parts (eyes, mouth)
        layers.Conv2D(128, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(128, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2,2),  # 24 → 12
        layers.Dropout(0.25),

        # Block 3 — detect high-level facial expressions
        layers.Conv2D(256, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2,2),  # 12 → 6
        layers.Dropout(0.25),

        # Classification head
        layers.Flatten(),
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5),
        layers.Dense(256),
        layers.Activation('relu'),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    return model

model_A = build_custom_cnn()
model_A.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model_A.summary()
print(f'\nModel A parameters: {model_A.count_params():,}')

In [ ]:
# ── MODEL B: TRANSFER LEARNING WITH VGG16 ────────────────────

# FER images are 48x48 grayscale, but VGG16 expects 48x48 RGB.
# We convert grayscale→RGB by repeating the channel 3 times.

def build_transfer_model():
    # Input: grayscale 48x48
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1))

    # Convert grayscale → 3-channel
    x = layers.Lambda(lambda img: tf.repeat(img, 3, axis=-1))(inputs)

    # Load VGG16 — include_top=False removes the final dense layers
    # weights='imagenet' loads pre-trained weights
    base_model = VGG16(
        include_top=False,
        weights='imagenet',
        input_tensor=x
    )

    for layer in base_model.layers:
        layer.trainable = False

    for layer in base_model.layers[-4:]:
        layer.trainable = True

    # Build classification head on top of VGG16 features
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)  # Better than Flatten for transfer learning
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model, base_model

model_B, base_vgg = build_transfer_model()
model_B.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),  # Lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trainable = sum(1 for l in model_B.layers if l.trainable)
frozen    = sum(1 for l in model_B.layers if not l.trainable)
print(f'Model B — Trainable layers: {trainable}, Frozen layers: {frozen}')
print(f'Model B parameters: {model_B.count_params():,}')

## Step 6 — Train Both Models

In [ ]:
# ── SHARED CALLBACKS ─────────────────────────────────────────
def get_callbacks(model_name):
    return [
        callbacks.ReduceLROnPlateau(
            monitor='val_accuracy', factor=0.5,
            patience=4, min_lr=1e-7, verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_accuracy', patience=10,
            restore_best_weights=True, verbose=1
        ),
        callbacks.ModelCheckpoint(
            f'best_{model_name}.keras',
            monitor='val_accuracy',
            save_best_only=True, verbose=0
        )
    ]

In [ ]:
# ── TRAIN MODEL A: CUSTOM CNN (~15–20 min) ───────────────────
print('Training Model A — Custom CNN')
print('Expected time: ~15–20 minutes on T4 GPU\n')

history_A = model_A.fit(
    train_generator,
    epochs=50,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('model_A'),
    verbose=1
)

loss_A, acc_A = model_A.evaluate(test_generator, verbose=0)
print(f'\nModel A — Test Accuracy: {acc_A*100:.2f}%')

In [ ]:
# ── TRAIN MODEL B: TRANSFER LEARNING (~10–15 min) ────────────
print('Training Model B — VGG16 Transfer Learning')
print('Expected time: ~10–15 minutes on T4 GPU\n')

history_B = model_B.fit(
    train_generator,
    epochs=50,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('model_B'),
    verbose=1
)

loss_B, acc_B = model_B.evaluate(test_generator, verbose=0)
print(f'\nModel B — Test Accuracy: {acc_B*100:.2f}%')

In [ ]:
# ── COMPARE & SELECT BEST MODEL ──────────────────────────────
print('\n' + '='*45)
print('  MODEL COMPARISON')
print('='*45)
print(f'  Model A (Custom CNN)         : {acc_A*100:.2f}%')
print(f'  Model B (VGG16 Transfer)     : {acc_B*100:.2f}%')
print('='*45)

if acc_B >= acc_A:
    best_model = model_B
    best_history = history_B
    print('  Winner: Model B (Transfer Learning) ✓')
else:
    best_model = model_A
    best_history = history_A
    print('  Winner: Model A (Custom CNN) ✓')
print('='*45)

## Step 7 — Evaluate: Plots + Confusion Matrix

In [ ]:
# ── PLOT TRAINING CURVES FOR BOTH MODELS ─────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training History — Model A (Custom CNN) vs Model B (VGG16)', fontsize=13, fontweight='bold')

for i, (hist, name, color) in enumerate([
    (history_A, 'Custom CNN', 'royalblue'),
    (history_B, 'VGG16 Transfer', 'darkorange')
]):
    ep = range(1, len(hist.history['accuracy']) + 1)

    axes[i,0].plot(ep, hist.history['accuracy'],     label='Train',      color=color,      linewidth=2)
    axes[i,0].plot(ep, hist.history['val_accuracy'], label='Validation', color=color,      linewidth=2, linestyle='--')
    axes[i,0].set_title(f'{name} — Accuracy')
    axes[i,0].set_ylabel('Accuracy')
    axes[i,0].set_xlabel('Epoch')
    axes[i,0].legend()
    axes[i,0].grid(True, alpha=0.3)

    axes[i,1].plot(ep, hist.history['loss'],     label='Train',      color=color,      linewidth=2)
    axes[i,1].plot(ep, hist.history['val_loss'], label='Validation', color=color,      linewidth=2, linestyle='--')
    axes[i,1].set_title(f'{name} — Loss')
    axes[i,1].set_ylabel('Loss')
    axes[i,1].set_xlabel('Epoch')
    axes[i,1].legend()
    axes[i,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CONFUSION MATRIX ON TEST SET ─────────────────────────────
test_generator.reset()
y_pred_probs = best_model.predict(test_generator, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = test_generator.classes

cm = confusion_matrix(y_true, y_pred)

# Normalise to percentages
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Confusion Matrix — Best Model on Test Set', fontsize=13, fontweight='bold')

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS,
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Raw Counts')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=45)

# Normalised %
sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Greens',
            xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS,
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Normalised (% per class)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPer-Class Report:')
print(classification_report(y_true, y_pred, target_names=EMOTION_LABELS))

In [ ]:
# ── VISUALISE PREDICTIONS WITH CONFIDENCE ────────────────────

test_generator.reset()
batch_imgs, batch_labels = next(test_generator)
batch_preds = best_model.predict(batch_imgs, verbose=0)

fig, axes = plt.subplots(2, 7, figsize=(16, 6))
fig.suptitle('Predictions on Test Images — Green=Correct, Red=Wrong', fontsize=12, fontweight='bold')

for i in range(14):
    row, col = i // 7, i % 7
    true_idx = np.argmax(batch_labels[i])
    pred_idx = np.argmax(batch_preds[i])
    confidence = batch_preds[i][pred_idx] * 100
    correct = true_idx == pred_idx

    axes[row,col].imshow(batch_imgs[i,:,:,0], cmap='gray')
    axes[row,col].set_title(
        f'T: {EMOTION_LABELS[true_idx]}\nP: {EMOTION_LABELS[pred_idx]} ({confidence:.0f}%)',
        fontsize=7,
        color='green' if correct else 'red'
    )
    axes[row,col].axis('off')

plt.tight_layout()
plt.savefig('predictions_sample.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8 — Real-Time Webcam Emotion Detection
This runs live emotion detection on your webcam using OpenCV's Haar Cascade face detector.

**Note:** Webcam access works in a local Jupyter setup.  
In Colab, we use a JavaScript-based approach to capture a single frame from webcam and run prediction on it.

In [ ]:
# ── COLAB WEBCAM CAPTURE + SINGLE FRAME PREDICTION ───────────
# This captures one photo from your webcam and runs emotion detection on it.

from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js
from base64 import b64decode
import io
from PIL import Image

def take_photo():
    """Capture a frame from webcam using JavaScript in Colab."""
    js = Javascript('''
        async function takePhoto() {
            const div = document.createElement('div');
            const video = document.createElement('video');
            const button = document.createElement('button');
            button.textContent = 'Capture Emotion';
            button.style.cssText = 'padding:10px 20px; font-size:16px; cursor:pointer; margin-top:10px;';
            div.appendChild(video);
            div.appendChild(button);
            document.body.appendChild(div);

            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await video.play();

            await new Promise(resolve => button.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getTracks().forEach(t => t.stop());
            div.remove();
            return canvas.toDataURL('image/jpeg', 0.8);
        }
        takePhoto();
    ''')
    display(js)
    data = eval_js('takePhoto()')
    binary = b64decode(data.split(',')[1])
    return np.array(Image.open(io.BytesIO(binary)))

print('Run the next cell to open your webcam and capture a photo for emotion detection.')

In [ ]:
# ── DETECT EMOTION FROM WEBCAM PHOTO ─────────────────────────
# Download Haar Cascade face detector
!wget -q https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml
face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

def detect_and_predict(frame_rgb):
    """Detect faces in frame, predict emotion for each face."""
    frame_bgr  = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
    frame_gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)

    # Detect faces using Haar Cascade
    faces = face_cascade.detectMultiScale(
        frame_gray, scaleFactor=1.1,
        minNeighbors=5, minSize=(30, 30)
    )

    results = []
    output_frame = frame_rgb.copy()

    for (x, y, w, h) in faces:
        # Crop face region
        face_roi = frame_gray[y:y+h, x:x+w]

        # Preprocess for model: resize to 48x48, normalise
        face_resized = cv2.resize(face_roi, (IMG_SIZE, IMG_SIZE))
        face_input   = face_resized.astype('float32') / 255.0
        face_input   = face_input.reshape(1, IMG_SIZE, IMG_SIZE, 1)

        # Predict
        probs       = best_model.predict(face_input, verbose=0)[0]
        pred_idx    = np.argmax(probs)
        emotion     = EMOTION_LABELS[pred_idx]
        confidence  = probs[pred_idx] * 100

        results.append((emotion, confidence, probs))

        # Draw bounding box and label on frame
        color = (0, 255, 0)  # Green box
        cv2.rectangle(output_frame, (x,y), (x+w, y+h), color, 2)
        label = f'{emotion}: {confidence:.1f}%'
        cv2.putText(output_frame, label, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    return output_frame, results, len(faces)


# Capture webcam photo
print('A webcam window will appear. Make a facial expression and click "Capture Emotion".')
photo = take_photo()

# Run detection
output_frame, results, num_faces = detect_and_predict(photo)

# Display result
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Real-Time Emotion Detection — {num_faces} face(s) detected', fontsize=12, fontweight='bold')

axes[0].imshow(output_frame)
axes[0].set_title('Detected Face with Emotion Label')
axes[0].axis('off')

if results:
    emotion, confidence, probs = results[0]
    colors_bar = ['#E24B4A' if i == np.argmax(probs) else '#B5D4F4' for i in range(NUM_CLASSES)]
    axes[1].barh(EMOTION_LABELS, probs * 100, color=colors_bar, edgecolor='gray', linewidth=0.5)
    axes[1].set_title(f'Prediction: {emotion} ({confidence:.1f}%)')
    axes[1].set_xlabel('Confidence (%)')
    axes[1].axvline(x=50, color='gray', linestyle='--', alpha=0.5)
    axes[1].grid(True, axis='x', alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'No face detected.\nTry better lighting or move closer.',
                 ha='center', va='center', fontsize=12, transform=axes[1].transAxes)
    axes[1].axis('off')

plt.tight_layout()
plt.savefig('webcam_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── LOCAL JUPYTER: REAL-TIME VIDEO LOOP ──────────────────────

def run_realtime_webcam():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print('Could not open webcam.')
        return

    print('Webcam opened. Press Q to quit.')
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        output_frame, results, _ = detect_and_predict(frame_rgb)
        output_bgr = cv2.cvtColor(output_frame, cv2.COLOR_RGB2BGR)

        cv2.imshow('Emotion Detector — Press Q to quit', output_bgr)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print('Webcam closed.')

# Uncomment the line below to run locally:
# run_realtime_webcam()

## Step 9 — Save Model & Project Summary

In [ ]:
# Save best model
best_model.save('emotion_detector_final.keras')
print('Model saved as emotion_detector_final.keras')

# Download model file to your computer
from google.colab import files
files.download('emotion_detector_final.keras')

# Final summary
best_acc = max(acc_A, acc_B)
winner   = 'VGG16 Transfer Learning' if acc_B >= acc_A else 'Custom CNN'

print('\n' + '='*52)
print('  PROJECT SUMMARY — FACIAL EMOTION DETECTOR')
print('='*52)
print(f'  Dataset         : FER-2013 ({total_train} train / {total_test} test)')
print(f'  Classes         : 7 emotions')
print(f'  Best Architecture : {winner}')
print(f'  Test Accuracy   : {best_acc*100:.2f}%')
print(f'  Key Techniques  : Transfer Learning (VGG16), Class Weights,')
print(f'                    Data Augmentation, Haar Cascade Detection,')
print(f'                    BatchNorm, Dropout, ReduceLROnPlateau')
print(f'  Outputs         : sample_emotions.png, class_distribution.png,')
print(f'                    training_curves.png, confusion_matrix.png,')
print(f'                    predictions_sample.png, webcam_prediction.png')
print('='*52)